# 00 · Setup
Creates the Medallion schemas and the landing volume in Unity Catalog.
Safe to re-run: every statement uses IF NOT EXISTS.

In [0]:
# A widget makes the catalog a parameter instead of a hard-coded value,
# so the same notebook works in dev and prod catalogs.
dbutils.widgets.text("catalog", "workspace")
catalog = dbutils.widgets.get("catalog")
print(f"Using catalog: {catalog}")

In [0]:
layers = {
    "sap_bronze": "Raw SAP extracts, append-only. Never transformed, so it can be replayed.",
    "sap_silver": "Cleaned, typed SAP data with CDC applied and SCD2 history.",
    "sap_gold":   "Star schema and aggregates for Power BI and Genie.",
}

for schema, comment in layers.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema} COMMENT '{comment}'")
    print(f"ok  {catalog}.{schema}")

In [0]:
# A managed volume = file storage governed by Unity Catalog (permissions, lineage).
# It replaces the old DBFS mounts. Raw files live next to the raw layer.
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {catalog}.sap_bronze.landing
    COMMENT 'Landing zone where SAP extract files arrive'
""")

landing_path = f"/Volumes/{catalog}/sap_bronze/landing"
dbutils.fs.mkdirs(f"{landing_path}/sap")
print(f"Landing path: {landing_path}/sap")

In [0]:
display(spark.sql(f"SHOW SCHEMAS IN {catalog} LIKE 'sap_*'"))
display(spark.sql(f"SHOW VOLUMES IN {catalog}.sap_bronze"))